# Weather - Silver Layer Transformation

**Purpose:** Transform weather data from bronze to silver layer with cleaning, enrichment, and validation

**Bronze Table:** `vattenfall_dev.raw.bronze_weather`

**Silver Table:** `vattenfall_dev.refined.silver_weather`

**Transformations Applied:**
- Measurement type casting (temperature, wind speed, cloud cover, precipitation)
- Region normalization (city names to country codes)
- Alert-level normalization (standardized warning levels)
- Impossible weather values filtering
- Day-level date fields (report_day)
- Reusable weather indicators (heating/cooling demand, wind power potential, renewable energy favorability)
- Temporal features and rolling statistics

In [0]:
import sys
import os

# Add project root to Python path
project_root = "/Workspace/Users/gharbi@startsteps.org/vattenfall-capstone-project"

if project_root not in sys.path:
    sys.path.append(project_root)

print(f"✅ Added to Python path: {project_root}")

In [0]:
# Import transformation functions
from src.transforms import weather_transforms

# Import validation functions
from src.validation.silver_checks import (
    SilverDataValidator,
    validate_weather_data
)

# Import PySpark functions
from pyspark.sql import functions as F

print("✅ All modules imported successfully")
print("\nAvailable weather transformation functions:")
for func in dir(weather_transforms):
    if not func.startswith('_') and callable(getattr(weather_transforms, func)):
        print(f"  - {func}()")

In [0]:
import yaml

# Load project configuration
config_path = f"{project_root}/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Extract configuration values
catalog = config["catalog"]
raw_schema = config["schemas"]["raw"]
refined_schema = config["schemas"]["refined"]

# Define table names
bronze_table = f"{catalog}.{raw_schema}.bronze_weather"
silver_table = f"{catalog}.{refined_schema}.silver_weather"

print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
print(f"Catalog:        {catalog}")
print(f"Raw Schema:     {raw_schema}")
print(f"Refined Schema: {refined_schema}")
print(f"\nBronze Table:   {bronze_table}")
print(f"Silver Table:   {silver_table}")
print("=" * 70)

In [0]:
# Load bronze weather data
df_bronze = spark.table(bronze_table)

print("=" * 70)
print("BRONZE DATA SUMMARY")
print("=" * 70)
print(f"Total Records: {df_bronze.count():,}")
print(f"Total Columns: {len(df_bronze.columns)}")
print("=" * 70)

print("\nBronze Schema:")
df_bronze.printSchema()

print("\nBronze Sample Data:")
display(df_bronze.limit(5))

In [0]:
# Apply complete transformation pipeline from bronze to silver
print("Starting transformation pipeline...\n")

df_silver = weather_transforms.transform_bronze_to_silver(df_bronze)

# Drop unused columns (100% nulls from Bronze)
columns_to_drop = ["event_date", "weather_alert_level", "source_system", "last_updated_ts", "_rescued_data"]
existing_cols_to_drop = [col for col in columns_to_drop if col in df_silver.columns]

if existing_cols_to_drop:
    df_silver = df_silver.drop(*existing_cols_to_drop)
    print(f"✅ Dropped unused columns: {', '.join(existing_cols_to_drop)}")

print("✅ Transformations applied successfully")
print("\n" + "=" * 70)
print("TRANSFORMATION SUMMARY")
print("=" * 70)
print(f"Bronze Records:  {df_bronze.count():,}")
print(f"Silver Records:  {df_silver.count():,}")
print(f"Records Dropped: {df_bronze.count() - df_silver.count():,}")
print(f"\nBronze Columns:  {len(df_bronze.columns)}")
print(f"Silver Columns:  {len(df_silver.columns)}")
print("=" * 70)

In [0]:
print("Silver Layer Schema:")
df_silver.printSchema()

print("\n" + "=" * 70)
print("SILVER DATA SAMPLE")
print("=" * 70)

display(df_silver.limit(10))

print("\nNew Columns Added:")
new_columns = set(df_silver.columns) - set(df_bronze.columns)
for col in sorted(new_columns):
    print(f"  - {col}")

In [0]:
print("=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)

# Record counts
total_silver = df_silver.count()
print(f"\nTotal Silver Records: {total_silver:,}")

# Completeness check for key columns
key_columns = ["timestamp", "region_normalized", "temperature_c", "wind_speed_ms", "report_day"]

print("\nCompleteness Check (Key Columns):")
for col in key_columns:
    if col in df_silver.columns:
        null_count = df_silver.filter(F.col(col).isNull()).count()
        completeness_pct = ((total_silver - null_count) / total_silver * 100) if total_silver > 0 else 0
        status = "✅" if completeness_pct == 100 else "⚠️"
        print(f"  {status} {col:25s}: {completeness_pct:6.2f}% complete ({null_count:,} nulls)")

# Check for extreme conditions
if "is_extreme_temp" in df_silver.columns:
    extreme_temp_count = df_silver.filter(F.col("is_extreme_temp") == True).count()
    extreme_temp_pct = (extreme_temp_count / total_silver * 100) if total_silver > 0 else 0
    print(f"\nExtreme Temperature Events: {extreme_temp_count:,} ({extreme_temp_pct:.2f}%)")

if "is_high_wind" in df_silver.columns:
    high_wind_count = df_silver.filter(F.col("is_high_wind") == True).count()
    high_wind_pct = (high_wind_count / total_silver * 100) if total_silver > 0 else 0
    print(f"High Wind Events: {high_wind_count:,} ({high_wind_pct:.2f}%)")

# Data quality score distribution
if "data_quality_score" in df_silver.columns:
    print("\nData Quality Score Distribution:")
    df_silver.groupBy("data_quality_score").count().orderBy("data_quality_score", ascending=False).show()

print("=" * 70)

In [0]:
print(f"Writing silver data to {silver_table}...\n")

# Write to silver table with Delta format
df_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", "true") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table)

print("✅ Data successfully written to silver table")
print("\n" + "=" * 70)
print("WRITE OPERATION SUMMARY")
print("=" * 70)
print(f"Table:   {silver_table}")
print(f"Format:  Delta")
print(f"Mode:    Overwrite")
print(f"Records: {df_silver.count():,}")
print("=" * 70)

In [0]:
# Reload the validation module to pick up the fixes
import importlib
import src.validation.silver_checks
importlib.reload(src.validation.silver_checks)
from src.validation.silver_checks import SilverDataValidator, validate_weather_data

print("✅ Validation module reloaded with fixes")

In [0]:
print("Running data quality validation...\n")

# Load silver table for validation
df_silver_verify = spark.table(silver_table)

# Run pre-configured validation
print("=" * 70)
print("AUTOMATED VALIDATION CHECKS")
print("=" * 70)
results = validate_weather_data(df_silver_verify)

# Run custom validation with detailed report
print("\n" + "=" * 70)
print("DETAILED VALIDATION REPORT")
print("=" * 70)

validator = SilverDataValidator(df_silver_verify, silver_table)

# Completeness check
validator.check_completeness(["timestamp", "region_normalized", "temperature_c", "wind_speed_ms"])

# Duplicate check
validator.check_duplicates(["timestamp", "region_original"])

# Value range checks
validator.check_value_ranges({
    "temperature_c": (-50, 60),
    "wind_speed_ms": (0, 100),
    "cloud_cover_percent": (0, 100),
    "hour": (0, 23)
})

# Timeliness check
validator.check_timeliness("timestamp", max_age_hours=48)

# Generate and print report
print(validator.generate_report())

In [0]:
# Final verification and statistics
df_final = spark.table(silver_table)

print("=" * 70)
print("SILVER TABLE VERIFICATION")
print("=" * 70)
print(f"Table Name:    {silver_table}")
print(f"Total Records: {df_final.count():,}")
print(f"Total Columns: {len(df_final.columns)}")
print("=" * 70)

# Show descriptive statistics for weather measurements
print("\nWeather Measurement Statistics:")
df_final.select(
    "temperature_c",
    "wind_speed_ms",
    "cloud_cover_percent",
    "precipitation_mm"
).describe().show()

# Show regional distribution
print("\nRegional Distribution:")
df_final.groupBy("region_normalized", "region_original").count().orderBy("region_normalized", "count", ascending=False).show(20, False)

# Show alert level distribution
print("\nAlert Level Distribution:")
df_final.groupBy("alert_level_normalized").count().orderBy("count", ascending=False).show()

print("\n" + "=" * 70)
print("FINAL SAMPLE DATA")
print("=" * 70)
display(df_final.orderBy(F.desc("timestamp")).limit(20))

print("\n✅ Silver layer transformation complete!")
print(f"\nSilver table is ready for analytics: {silver_table}")

In [0]:
# Summary of key transformations applied
print("=" * 70)
print("KEY TRANSFORMATION RESULTS")
print("=" * 70)

# Region normalization results
print("\n1. REGION NORMALIZATION")
print("-" * 70)
df_final.groupBy("region_normalized", "region_original") \
    .agg(F.count("*").alias("record_count")) \
    .orderBy("region_normalized", "record_count", ascending=[True, False]) \
    .show(20, False)

# Alert level normalization
print("\n2. ALERT LEVEL NORMALIZATION")
print("-" * 70)
df_final.groupBy("alert_level_normalized") \
    .agg(F.count("*").alias("record_count")) \
    .orderBy("alert_level_normalized") \
    .show(10, False)

# Report day coverage
print("\n3. REPORT DAY COVERAGE")
print("-" * 70)
df_final.groupBy("report_day") \
    .agg(
        F.count("*").alias("hourly_records"),
        F.countDistinct("region_normalized").alias("regions"),
        F.round(F.avg("temperature_c"), 2).alias("avg_temp"),
        F.round(F.avg("wind_speed_ms"), 2).alias("avg_wind")
    ) \
    .orderBy("report_day") \
    .show(10, False)

# Weather indicators
print("\n4. WEATHER INDICATORS SUMMARY")
print("-" * 70)
print("\nTemperature Categories:")
df_final.groupBy("temp_category").count().orderBy("temp_category").show()

print("\nWind Categories:")
df_final.groupBy("wind_category").count().orderBy("wind_category").show()

print("\nPrecipitation Intensity:")
df_final.groupBy("precipitation_intensity").count().orderBy("precipitation_intensity").show()

print("\nRenewable Energy Favorable Conditions:")
df_final.groupBy("renewable_energy_favorable").count().show()

print("\n" + "=" * 70)
print("✅ All transformations validated successfully!")
print("=" * 70)